# Feature Selection and Preprocessing

## Objectives

1. Import modular feature sets.
2. Review selected variables.
3. Handle BRFSS special response codes.
4. Convert target variable to binary.
5. Apply keep/drop decisions.
6. Create cleaned modeling dataset.
7. Export processed dataset for modeling.

In [24]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

In [25]:
# Add project root to Python path so we can import from src
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.feature_sets import (
    get_target_variable,
    get_all_predictor_features,
    get_modeling_variables,
)

In [26]:
raw_data_path = PROJECT_ROOT / "data" / "raw" / "LLCP2024.XPT"

df = pd.read_sas(raw_data_path)

print("Dataset shape:", df.shape)

Dataset shape: (457670, 301)


In [27]:
target = get_target_variable()
predictor_features = get_all_predictor_features(include_additional=True)
modeling_vars = get_modeling_variables(include_additional=True)

print("Target:", target)
print("Number of predictor features:", len(predictor_features))
print("Number of total modeling variables:", len(modeling_vars))

missing_vars = [var for var in modeling_vars if var not in df.columns]
print("Missing variables:", missing_vars)

Target: MEDCOST1
Number of predictor features: 31
Number of total modeling variables: 32
Missing variables: []


In [28]:
model_df = df[modeling_vars].copy()

model_df.head()

,MEDCOST1,_STATE,SEXVAR,MARITAL,EDUCA,RENTHOM1,_AGE80,_RACEGR3,INCOME3,EMPLOY1,...,CHCSCNC1,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,EXERANY2,SMOKE100,_BMI5,LASTDEN4,RMVTETH4
0,2.0,1.0,2.0,3.0,4.0,1.0,78.0,1.0,99.0,7.0,...,1.0,2.0,2.0,2.0,2.0,1.0,2.0,2249.0,1.0,1.0
1,2.0,1.0,1.0,1.0,6.0,1.0,80.0,1.0,11.0,7.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2583.0,1.0,1.0
2,1.0,1.0,1.0,6.0,5.0,1.0,59.0,1.0,99.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2253.0,4.0,2.0
3,2.0,1.0,1.0,1.0,6.0,1.0,80.0,1.0,6.0,7.0,...,1.0,2.0,2.0,2.0,2.0,1.0,2.0,2509.0,1.0,8.0
4,2.0,1.0,1.0,5.0,5.0,1.0,47.0,1.0,3.0,8.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1977.0,1.0,1.0


## BRFSS Special Response Codes

BRFSS uses numeric codes for non-substantive responses. These codes need to be handled before modeling.

Common codes in this project include:

- 7 = Don't know / Not sure
- 9 = Refused
- 77 = Don't know / Not sure
- 88 = None / No days / No usual source, depending on the variable
- 99 = Refused / Missing

For most categorical variables, Don't Know and Refused responses will be converted to missing values. For day-count variables, 88 represents zero days and should be recoded to 0.

In [29]:
def replace_special_codes(series, missing_codes):
    """
    Replace BRFSS special response codes with NaN.
    """
    return series.replace(missing_codes, np.nan)


def recode_days_variable(series):
    """
    Recode BRFSS days variables:
    - 88 means zero days
    - 77 and 99 are non-substantive responses and become NaN
    """
    return series.replace({
        88: 0,
        77: np.nan,
        99: np.nan
    })


def recode_yes_no(series):
    """
    Recode BRFSS yes/no variables:
    - 1 = Yes
    - 2 = No
    - 7/9 = missing
    """
    return series.replace({
        1: 1,
        2: 0,
        7: np.nan,
        9: np.nan
    })

In [30]:
# MEDCOST1:
# 1 = Yes, could not see doctor due to cost
# 2 = No
# 7 = Don't know / Not sure
# 9 = Refused

model_df["MEDCOST1_binary"] = model_df["MEDCOST1"].replace({
    1: 1,
    2: 0,
    7: np.nan,
    9: np.nan
})

model_df["MEDCOST1_binary"].value_counts(dropna=False)

MEDCOST1_binary
0.0    412634
1.0     43363
NaN      1673
Name: count, dtype: int64

In [31]:
model_df = model_df.dropna(subset=["MEDCOST1_binary"]).copy()

model_df["MEDCOST1_binary"] = model_df["MEDCOST1_binary"].astype(int)

model_df["MEDCOST1_binary"].value_counts(normalize=True) * 100

MEDCOST1_binary
0    90.490508
1     9.509492
Name: proportion, dtype: float64

In [32]:
yes_no_vars = [
    "VETERAN3",
    "CVDINFR4",
    "CVDCRHD4",
    "CVDSTRK3",
    "ASTHMA3",
    "CHCSCNC1",
    "CHCOCNC1",
    "CHCCOPD3",
    "ADDEPEV3",
    "CHCKDNY2",
    "EXERANY2",
    "SMOKE100",
]

for var in yes_no_vars:
    model_df[var] = recode_yes_no(model_df[var])

In [33]:
day_vars = [
    "PHYSHLTH",
    "MENTHLTH",
    "POORHLTH",
]

for var in day_vars:
    model_df[var] = recode_days_variable(model_df[var])

In [34]:
categorical_missing_codes = {
    "MARITAL": [9],
    "EDUCA": [9],
    "RENTHOM1": [7, 9],
    "_RACEGR3": [9],
    "INCOME3": [77, 99],
    "EMPLOY1": [9],
    "PRIMINS2": [77, 99],
    "PERSDOC3": [7, 9],
    "CHECKUP1": [7, 9],
    "GENHLTH": [7, 9],
    "LASTDEN4": [7, 9],
    "RMVTETH4": [7, 9],
}

for var, codes in categorical_missing_codes.items():
    model_df[var] = replace_special_codes(model_df[var], codes)

In [35]:
# _BMI5 is stored as BMI * 100 in BRFSS
model_df["_BMI5"] = model_df["_BMI5"] / 100

model_df["_BMI5"].describe()

count    413354.000000
mean         28.557600
std           6.583895
min          12.000000
25%          24.140000
50%          27.440000
75%          31.750000
max          99.840000
Name: _BMI5, dtype: float64

In [36]:
# Drop original target because MEDCOST1_binary is the cleaned target.
# ASTHNOW is excluded because it is only asked of respondents with asthma history
# and overlaps conceptually with ASTHMA3.

drop_vars = [
    "MEDCOST1",
    "ASTHNOW",
]

model_df = model_df.drop(columns=[var for var in drop_vars if var in model_df.columns])

model_df.head()

,_STATE,SEXVAR,MARITAL,EDUCA,RENTHOM1,_AGE80,_RACEGR3,INCOME3,EMPLOY1,VETERAN3,...,CHCOCNC1,CHCCOPD3,ADDEPEV3,CHCKDNY2,EXERANY2,SMOKE100,_BMI5,LASTDEN4,RMVTETH4,MEDCOST1_binary
0,1.0,2.0,3.0,4.0,1.0,78.0,1.0,NaN,7.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,22.49,1.0,1.0,0
1,1.0,1.0,1.0,6.0,1.0,80.0,1.0,11.0,7.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,25.83,1.0,1.0,0
2,1.0,1.0,6.0,5.0,1.0,59.0,1.0,NaN,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,22.53,4.0,2.0,1
3,1.0,1.0,1.0,6.0,1.0,80.0,1.0,6.0,7.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,25.09,1.0,8.0,0
4,1.0,1.0,5.0,5.0,1.0,47.0,1.0,3.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,19.77,1.0,1.0,0


In [37]:
missing_after_recode = (
    model_df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_after_recode

POORHLTH           42.610368
INCOME3            19.011309
_BMI5               9.351597
SMOKE100            6.845440
PRIMINS2            3.989500
PHYSHLTH            2.360761
RMVTETH4            2.025891
_RACEGR3            1.957030
EMPLOY1             1.801547
MENTHLTH            1.729178
LASTDEN4            1.214701
CHECKUP1            1.131367
CVDCRHD4            0.974349
PERSDOC3            0.967770
MARITAL             0.901541
RENTHOM1            0.836190
CVDINFR4            0.657241
CHCSCNC1            0.641671
ADDEPEV3            0.557898
VETERAN3            0.542986
CHCOCNC1            0.518863
EDUCA               0.494740
CHCCOPD3            0.454828
CHCKDNY2            0.413599
ASTHMA3             0.390792
CVDSTRK3            0.301098
GENHLTH             0.272151
EXERANY2            0.263598
_STATE              0.000000
SEXVAR              0.000000
_AGE80              0.000000
MEDCOST1_binary     0.000000
dtype: float64

## Investigation of POORHLTH Missingness

In [38]:
df["POORHLTH"].value_counts(dropna=False).sort_index()

POORHLTH
1.0      12487
2.0      16082
3.0      10938
4.0       6132
5.0      13184
6.0       1919
7.0       6257
8.0       1583
9.0        298
10.0     10316
11.0       101
12.0       972
13.0       140
14.0      2696
15.0     10360
16.0       255
17.0       184
18.0       264
19.0        47
20.0      6572
21.0       639
22.0       132
23.0       105
24.0       119
25.0      2354
26.0        92
27.0       155
28.0       520
29.0       182
30.0     21013
77.0      4219
88.0    136544
99.0      1388
NaN     189421
Name: count, dtype: int64

In [39]:
df["POORHLTH"].isna().mean() * 100

np.float64(41.38811807634322)

In [40]:
pd.crosstab(
    model_df["MEDCOST1_binary"],
    model_df["POORHLTH"].isna(),
    normalize="index"
) * 100

POORHLTH,False,True
MEDCOST1_binary,,
0,55.095072,44.904928
1,79.224223,20.775777


In [41]:
model_df.groupby(
    model_df["POORHLTH"].isna()
)["MEDCOST1_binary"].mean() * 100

POORHLTH
False    13.127496
True      4.636597
Name: MEDCOST1_binary, dtype: float64

## POORHLTH Decision

POORHLTH measures the number of days during the past 30 days that poor physical or mental health limited a respondent's usual activities. Because this variable had substantial missingness, additional investigation was conducted to determine whether it should be retained for modeling.

### Findings

- POORHLTH contains a large number of missing values, requiring additional investigation before inclusion in the modeling dataset.
- BRFSS code 88 represents respondents reporting zero days of activity limitation and was recoded accordingly.
- Missing values therefore represent non-response rather than respondents reporting no activity limitations.
- Comparison of MEDCOST1 prevalence between respondents with missing and non-missing POORHLTH values showed substantial differences between the groups, indicating that respondents with missing values are not equivalent to respondents reporting zero days of activity limitation.
- POORHLTH measures health-related activity limitations and may be relevant to understanding barriers to healthcare access.

### Decision

**Retain POORHLTH for modeling.**

- Missing values were retained rather than recoded to zero because the analysis indicated that respondents with missing values differed meaningfully from respondents reporting zero days of activity limitation.

In [42]:
variable_decisions = pd.DataFrame([
    {"Variable": "MEDCOST1_binary", "Decision": "Keep", "Reason": "Cleaned binary target variable"},
    {"Variable": "_AGE80", "Decision": "Keep", "Reason": "Most granular age variable available; ages 80+ top-coded"},
    {"Variable": "_RACEGR3", "Decision": "Keep", "Reason": "Interpretable race/ethnicity grouping"},
    {"Variable": "ASTHMA3", "Decision": "Keep", "Reason": "Captures asthma diagnosis history"},
    {"Variable": "ASTHNOW", "Decision": "Drop", "Reason": "Skip logic variable; only asked if ASTHMA3 = Yes and adds preprocessing complexity"},
    {"Variable": "POORHLTH", "Decision": "Keep", "Reason": "Measures health-related activity limitations; missingness investigated and variable retained for modeling."},
    {"Variable": "_BMI5", "Decision": "Keep", "Reason": "Relevant health status feature; recoded from BMI*100 to BMI"},
    {"Variable": "INCOME3", "Decision": "Keep", "Reason": "Key socioeconomic predictor; special codes converted to missing"},
    {"Variable": "GENHLTH", "Decision": "Keep", "Reason": "Self-reported overall health status"},
    {"Variable": "CHECKUP1", "Decision": "Keep", "Reason": "Measures healthcare utilization"},
    {"Variable": "PERSDOC3", "Decision": "Keep", "Reason": "Access to a personal healthcare provider"},
    {"Variable": "LASTDEN4", "Decision": "Keep", "Reason": "Dental care utilization relevant to healthcare access"},
    {"Variable": "EMPLOY1", "Decision": "Keep", "Reason": "Employment status may influence insurance and affordability"},
    {"Variable": "_STATE", "Decision": "Keep", "Reason": "Included to account for state-level differences in healthcare access, policy, and cost environments."}
])

variable_decisions

,Variable,Decision,Reason
0,MEDCOST1_binary,Keep,Cleaned binary target variable
1,_AGE80,Keep,Most granular age variable available; ages 80+...
2,_RACEGR3,Keep,Interpretable race/ethnicity grouping
3,ASTHMA3,Keep,Captures asthma diagnosis history
4,ASTHNOW,Drop,Skip logic variable; only asked if ASTHMA3 = Y...
5,POORHLTH,Keep,Measures health-related activity limitations; ...
6,_BMI5,Keep,Relevant health status feature; recoded from B...
7,INCOME3,Keep,Key socioeconomic predictor; special codes con...
8,GENHLTH,Keep,Self-reported overall health status
9,CHECKUP1,Keep,Measures healthcare utilization


## Preprocessing Decisions

The target variable MEDCOST1 was converted to a binary outcome, where 1 indicates that the respondent could not see a doctor because of cost and 0 indicates no cost barrier. Don't Know, Refused, and missing target responses were removed.

BRFSS special response codes were handled based on variable type. Don't Know and Refused values were converted to missing values. For day-count variables, 88 was recoded as 0 days.

ASTHNOW was excluded from the initial cleaned modeling dataset because it is only asked of respondents who reported ever having asthma and overlaps conceptually with ASTHMA3.

POORHLTH was investigated due to high missingness and retained for modeling. Analysis showed that missing values do not represent respondents reporting zero days of activity limitation and that respondents with missing values differed from respondents with observed POORHLTH responses.

_BMI5 was converted from the BRFSS stored format into standard BMI units by dividing by 100.

_STATE was retained to capture potential state-level differences in healthcare policy, insurance environments, and healthcare access.

In [44]:
processed_data_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset.csv"
decision_table_path = PROJECT_ROOT / "outputs" / "feature_decision_table.csv"
missingness_path = PROJECT_ROOT / "outputs" / "missingness_after_recode.csv"

model_df.to_csv(processed_data_path, index=False)
variable_decisions.to_csv(decision_table_path, index=False)
missing_after_recode.to_csv(missingness_path)

print("Saved modeling dataset to:", processed_data_path)
print("Saved decision table to:", decision_table_path)
print("Saved missingness summary to:", missingness_path)
print("Final modeling dataset shape:", model_df.shape)

Saved modeling dataset to: /Users/Rechelle/Library/Mobile Documents/com~apple~CloudDocs/Grad School〽️/brfss-healthcare-access-analysis/data/processed/modeling_dataset.csv
Saved decision table to: /Users/Rechelle/Library/Mobile Documents/com~apple~CloudDocs/Grad School〽️/brfss-healthcare-access-analysis/outputs/feature_decision_table.csv
Saved missingness summary to: /Users/Rechelle/Library/Mobile Documents/com~apple~CloudDocs/Grad School〽️/brfss-healthcare-access-analysis/outputs/missingness_after_recode.csv
Final modeling dataset shape: (455997, 32)


# Summary

This notebook created the initial cleaned modeling dataset for the BRFSS healthcare access project.

Completed steps:
- Imported modular feature sets from `src/feature_sets.py`
- Loaded selected candidate variables
- Converted MEDCOST1 into a binary target variable
- Removed invalid target responses
- Recoded BRFSS special response codes
- Converted BMI into standard units
- POORHLTH missingness was investigated and the variable was retained for modeling.
- Excluded ASTHNOW from the initial modeling dataset
- Saved the cleaned dataset to `data/processed/modeling_dataset.csv`

The processed dataset is ready for supervised modeling.